# Titanic Dataset Feature Engineering and Selection

## Objective
The objective of this project is to prepare the Titanic dataset for machine learning by performing data cleaning, feature engineering, feature transformations, and feature selection. These steps improve the quality of the dataset and help identify the most important variables for predicting passenger survival.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

train.head()


## Loading the dataset
The training dataset contains the `Survived` column, which is the target variable. The test dataset is mainly for optional prediction work and does not include the target.

In [ ]:
train.shape
train.columns
train.info()
train.describe(include='all')
train.isnull().sum()

In [ ]:
train.isnull().sum().sort_values(ascending=False)
train["AgeMissing"] = train["Age"].isnull().astype(int)
train["Age"] = train["Age"].fillna(train["Age"].median())
train["Embarked"] = train["Embarked"].fillna(train["Embarked"].mode()[0])
train["Fare"] = train["Fare"].fillna(train["Fare"].median())
train["CabinMissing"] = train["Cabin"].isnull().astype(int)

### Missing Value Handling
- `Age` was imputed using the median because it is a numerical variable and median is robust to outliers.
- `Embarked` was imputed using the mode because it is categorical and has few missing values.
- `Cabin` has many missing values, so a missing indicator was created and deck information was extracted later rather than filling the raw values directly.

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=train["Fare"])
plt.title("Boxplot of Fare")
plt.show()

plt.figure(figsize=(8,4))
sns.boxplot(x=train["Age"])
plt.title("Boxplot of Age")
plt.show()

In [ ]:
Q1 = train["Fare"].quantile(0.25)
Q3 = train["Fare"].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
lower_bound = Q1 - 1.5 * IQR

In [ ]:
train["Fare"] = np.where(train["Fare"] > upper_bound, upper_bound, train["Fare"])

### Outlier Handling
Outliers were examined in `Fare` and `Age` using boxplots. `Fare` showed strong skewness and extreme high values, so outlier capping was applied using the IQR method. This helps reduce the influence of extreme ticket prices without removing valid passengers from the dataset.

In [ ]:
train["Sex"] = train["Sex"].str.strip().str.lower()
train["Embarked"] = train["Embarked"].astype(str).str.strip().str.upper()
train["Sex"].unique()
train["Embarked"].unique()

train.duplicated().sum()
train = train.drop_duplicates()

In [ ]:
### Data Consistency
Text-based categorical variables were standardized to avoid hidden inconsistencies caused by case or spacing differences. Duplicate rows were also checked and removed where necessary.

In [ ]:
train.to_csv("data/train_cleaned.csv", index=False)